In [2]:
import numpy as np
import pandas as pd
import glob
import xarray as xr
import os

#set current working directory
os.chdir('/home/jovyan')

#list files in shared folder
files = glob.glob("/shared_space/Galveston_Bay/*")
files


['/shared_space/Galveston_Bay/water_temp',
 '/shared_space/Galveston_Bay/MODIS_500',
 '/shared_space/Galveston_Bay/samples.csv',
 '/shared_space/Galveston_Bay/All_Observed_Data.csv',
 '/shared_space/Galveston_Bay/Observed_Raw',
 '/shared_space/Galveston_Bay/MODIS_test',
 '/shared_space/Galveston_Bay/MODIS_250',
 '/shared_space/Galveston_Bay/targets.csv',
 '/shared_space/Galveston_Bay/MODIS_files',
 '/shared_space/Galveston_Bay/feature_corr_new.png',
 '/shared_space/Galveston_Bay/feature_corr.png',
 '/shared_space/Galveston_Bay/ERA5']

In [6]:
observed_file_path='/shared_space/Galveston_Bay/All_Observed_Data.csv'
obs=pd.read_csv(observed_file_path)
obs.head(3)

,DateTime,Value,lat,lon,source
0,2010-01-04 00:00:00,12.0,29.295779,-94.918220,Stream Team
1,2010-01-04 00:00:00,10.2,29.503736,-95.098957,Stream Team
2,2010-01-05 09:02:00,9.7,29.723127,-94.942790,Regional Monitoring Database


In [3]:
# Read in Modis
modis_2019 = xr.open_dataset('/shared_space/Galveston_Bay/MODIS_test/surf_refl_250m_2019.nc')

In [4]:
#Read in Era5
era5 = xr.open_dataset("/shared_space/Galveston_Bay/ERA5/ERA5-2019-2020_v3_rescaled.nc")
era5

<xarray.Dataset>
Dimensions:     (latitude: 16, longitude: 21, valid_time: 17544)
Coordinates:
    number      int64 ...
  * valid_time  (valid_time) datetime64[ns] 2019-01-01 ... 2020-12-31T23:00:00
  * latitude    (latitude) float64 28.5 28.6 28.7 28.8 ... 29.7 29.8 29.9 30.0
  * longitude   (longitude) float64 -96.0 -95.9 -95.8 ... -94.2 -94.1 -94.0
Data variables:
    u10         (valid_time, latitude, longitude) float64 ...
    v10         (valid_time, latitude, longitude) float64 ...
    wind_speed  (valid_time, latitude, longitude) float64 ...
    ssr         (valid_time, latitude, longitude) float64 ...
    t2m         (valid_time, latitude, longitude) float64 ...

In [10]:
modis_2019 #red = b01 and nir= b02

<xarray.Dataset>
Dimensions:       (lat: 721, lon: 961, time: 365)
Coordinates:
  * lat           (lat) float32 30.002083 30.0 29.997917 ... 28.504168 28.502083
  * lon           (lon) float32 -96.00208 -96.0 ... -94.004166 -94.00208
  * time          (time) datetime64[ns] 2019-01-01 2019-01-02 ... 2019-12-31
Data variables:
    sur_refl_b01  (time, lat, lon) float32 ...
    sur_refl_b02  (time, lat, lon) float32 ...

Want to make a dataset that has feautres for all lat, lons to predict across. Uses code from preprocess.ipynb

In [14]:
print("ERA coords:", list(era5.coords))
print("MODIS coords:", list(modis_2019.coords))
print("ERA times:", era5.valid_time.min().values, "->", era5.valid_time.max().values)
print("MODIS times:", modis_2019.time.min().values, "->", modis_2019.time.max().values)

ERA coords: ['number', 'valid_time', 'latitude', 'longitude']
MODIS coords: ['lat', 'lon', 'time']
ERA times: 2019-01-01T00:00:00.000000000 -> 2020-12-31T23:00:00.000000000
MODIS times: 2019-01-01T00:00:00.000000000 -> 2019-12-31T00:00:00.000000000


In [17]:
#find average values for each day so that modis and era5 are on the same timescale
era_daily = era5.resample(valid_time='1D').mean(dim='valid_time')

/opt/conda/lib/python3.7/site-packages/xarray/core/nanops.py:160: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis=axis, dtype=dtype)


In [ ]:
#match the modis grids to its nearest era5 data point

#max distance between modis grid and era5
tolerance_deg = 0.03  # ~ 3 km; adjust up if grids not tightly overlapping

# interp requires the target coords to be 1D arrays
target_lat = era_daily['latitude']
target_lon = era_daily['longitude']

# If modis lat/lon are increasing/decreasing differently, interp handles it.
modis_on_era = modis_2019.interp(lat=target_lat, lon=target_lon, method='nearest')

# Now modis_on_era has dims (time, lat, lon) where lat/lon match ERA grid.
# If your modis time range differs, restrict to the intersection:
start = max(modis_on_era.time.min().values, era_daily.time.min().values)
end   = min(modis_on_era.time.max().values, era_daily.time.max().values)
modis_on_era = modis_on_era.sel(time=slice(start, end))
era_daily     = era_daily.sel(time=slice(start, end))

In [21]:
# Suppose era_daily is your xarray.Dataset
df = era_daily.to_dataframe().reset_index()
print(df.head())


   latitude  longitude valid_time  number       u10       v10  wind_speed  \
0      28.5      -96.0 2019-01-01       0 -2.197027 -3.736568    4.455292   
1      28.5      -96.0 2019-01-02       0 -2.537600 -6.781738    7.453773   
2      28.5      -96.0 2019-01-03       0  0.496403 -5.693661    8.675432   
3      28.5      -96.0 2019-01-04       0  4.679899 -1.794902    5.574112   
4      28.5      -96.0 2019-01-05       0 -0.175112  1.505580    1.827040   

             ssr         t2m  
0  382314.666667  288.055069  
1  115653.333333  287.452077  
2  237917.333333  285.555155  
3  573986.666667  286.161517  
4  585314.666667  288.769541  
